In [1]:
# ============================================================
# S6c: DOMAIN-AWARE REPRESENTATION ALIGNMENT (MMD)
#
# Builds on S5 (context-window student, best result so far:
# Acc=65.98%, F1=0.5974, N1=0.3514).
#
# MOTIVATION: KD (S2-S5) only aligns the student's OUTPUT
# distribution (soft labels) with the teacher's. It says nothing
# about whether the student's INTERNAL REPRESENTATION lives in a
# similar geometric space to the teacher's. PSG (clean, 3-channel
# clinical signal) and wearable (Zmax+E4, noisier, cheaper
# sensors) are different "domains" -- their raw signal statistics
# differ substantially. This script adds an explicit
# REPRESENTATION-LEVEL alignment term using Maximum Mean
# Discrepancy (MMD), which:
#   - measures the distance between the DISTRIBUTION of student
#     embeddings and the DISTRIBUTION of teacher embeddings
#     (computed per training batch)
#   - is minimized alongside the existing CE + KD losses
#   - requires no extra discriminator network (unlike adversarial
#     domain adaptation / GRL), making it far more stable to train
#     and much less hyperparameter-sensitive
#
#   Total loss = CE (artifact-weighted, same as S4/S5)
#              + KD_LAMBDA * KD (same as S5)
#              + MMD_LAMBDA * MMD(student_embedding, teacher_embedding)
#
# The student's fused embedding (192-dim, concat of z_ctx+e_ctx)
# is projected down to the teacher's embedding dimension (128)
# via a small linear layer before computing MMD, since MMD
# requires both feature sets to have matching dimensionality.
# This projection head is used ONLY for the alignment loss and
# does not affect the main classifier pathway.
#
# At inference, only the main classifier is used, exactly as in
# S5 -- the alignment projection head is discarded.
#
# Uses the 3-way split, VAL-selected checkpoints, TEST evaluated
# once -- same honest protocol as S1-S5.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
PSG_EEG_PATH      = r"D:\22\AA\preprocess\preprocessed_FFinal"
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"
SPLIT_PATH        = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"
TEACHER_CKPT_DIR  = r"D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed"

# ============================================================
# MECHANISM-SPECIFIC HYPERPARAMETERS
# ============================================================
MMD_LAMBDA     = 0.1          # weight on the MMD alignment term
MMD_KERNEL_MUL = 2.0          # multi-kernel MMD bandwidth multiplier
MMD_KERNEL_NUM = 5            # number of Gaussian kernels (multi-scale MMD)

# ============================================================
# CONTEXT CONFIG -- same as your best S5 run
# ============================================================
LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]

TEACHER_CONTEXT = 7
TEACHER_WINDOW  = 2 * TEACHER_CONTEXT + 1   # 15, fixed -- matches trained teacher

STUDENT_CONTEXT = 5     # matches your best S5 result
STUDENT_WINDOW  = 2 * STUDENT_CONTEXT + 1

EVAL_PATH = rf"D:\22\AA\AA journal\evaluation\teacher-student\s6c_domain_aware_mmd_studentC{STUDENT_CONTEXT}"
os.makedirs(EVAL_PATH, exist_ok=True)

N_EPOCHS    = 30
BATCH_SIZE  = 32
SEEDS         = [42, 123, 256, 789, 999]
TEACHER_SEEDS = [42, 123, 256, 789, 999]

T_D_MODEL = 128
T_DROPOUT = 0.4
T_N_HEADS = 4

S_D_MODEL = 96
S_DROPOUT = 0.3

KD_LAMBDA      = 0.5     # best value found in the S2/S4/S5 sweep
KD_TEMPERATURE = 4.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"S6c: DOMAIN-AWARE REPRESENTATION ALIGNMENT (MMD)")
print(f"Student context: +-{STUDENT_CONTEXT} (window={STUDENT_WINDOW})")
print(f"KD lambda     : {KD_LAMBDA}   Temperature: {KD_TEMPERATURE}   MMD lambda: {MMD_LAMBDA}")
print(f"Output        : {EVAL_PATH}")


# ============================================================
# ============  TEACHER ARCHITECTURE (frozen)  ================
# NOTE: adds forward_with_feature() to also expose the
# pre-classifier "center" representation, needed for MMD.
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    def __init__(self, in_ch=3, d_model=T_D_MODEL, dropout=T_DROPOUT):
        super().__init__()
        mid = d_model // 4

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)
        self.medium = time_branch(50)
        self.large  = time_branch(100)

        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)
        feat = self.se(feat)
        return self.proj(feat)


class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=T_D_MODEL, n_heads=T_N_HEADS, dropout=T_DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            d_model, d_model // 2, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)
        attn_out = self.transformer(x)
        return self.norm2(x + attn_out)


class MultiScaleSleepNetPlain(nn.Module):
    """Teacher architecture -- identical to your trained checkpoints.
    forward() is unchanged (returns logits only, for the frozen
    teacher_soft_targets() helper). forward_with_feature() is NEW
    and additionally returns the pre-classifier center embedding,
    used only for the MMD alignment term."""
    def __init__(self, in_ch=3, d_model=T_D_MODEL, n_layers=2,
                 dropout=T_DROPOUT, n_classes=5,
                 context=TEACHER_CONTEXT, window=TEACHER_WINDOW):
        super().__init__()
        self.context = context
        self.d_model = d_model
        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)
        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.inter_pos = nn.Parameter(torch.randn(1, window, d_model) * 0.01)
        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout) for _ in range(n_layers)
        ])
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes),
        )

    def _center_feature(self, x):
        B, W, C, T = x.shape
        cnn_out = self.cnn(x.view(B * W, C, T)).permute(0, 2, 1)
        intra = self.intra_blocks(cnn_out)
        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)
        inter = self.inter_blocks(epoch_feat + self.inter_pos)
        center = inter[:, self.context, :]
        return center

    def forward(self, x):
        center = self._center_feature(x)
        return self.classifier(center)

    def forward_with_feature(self, x):
        center = self._center_feature(x)
        return self.classifier(center), center


# ============================================================
# ========  STUDENT: PER-EPOCH ENCODER (same as S5)  ============
# Adds an alignment projection head (fused_embedding -> teacher
# embedding dim) used ONLY for the MMD loss.
# ============================================================
class ZmaxEEGEncoder(nn.Module):
    def __init__(self, in_ch=2, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        mid = d_model // 2

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2), nn.Dropout(dropout)
            )
        self.small, self.large = branch(15), branch(60)
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 1920)
            L_s, L_l = self.small(dummy).shape[2], self.large(dummy).shape[2]
        target_L = min(L_s, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.proj = nn.Sequential(nn.Conv1d(2 * mid, d_model, kernel_size=1),
                                   nn.BatchNorm1d(d_model), nn.GELU())
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (N, 2, 1920) where N = B*W (flattened batch*window)
        fs, fl = self.pool_s(self.small(x)), self.pool_l(self.large(x))
        feat = self.proj(torch.cat([fs, fl], dim=1)).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class E4Encoder(nn.Module):
    def __init__(self, in_ch=3, d_model=S_D_MODEL, dropout=S_DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, d_model // 2, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(d_model // 2), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model // 2, d_model, kernel_size=8, padding=4),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        feat = self.conv(x).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class InterEpochAggregator(nn.Module):
    def __init__(self, d_model=S_D_MODEL, context=STUDENT_CONTEXT):
        super().__init__()
        self.context = context
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        gru_out, _ = self.gru(x)
        contextualized = self.norm(x + gru_out)
        return contextualized[:, self.context, :]


class DomainAlignedStudentModel(nn.Module):
    """
    Same backbone as S5 (per-epoch encoders + inter-epoch BiGRU
    aggregators + concat fusion -> main classifier), PLUS a small
    alignment projection head that maps the fused embedding
    (2*S_D_MODEL=192-dim) down to the teacher's embedding
    dimension (T_D_MODEL=128-dim). This projected embedding is
    used ONLY for the MMD alignment loss during training.

    At inference, only `main_logits` matters -- the alignment
    head is discarded, identical cost/architecture to S5.
    """
    def __init__(self, d_model=S_D_MODEL, n_classes=5, dropout=S_DROPOUT,
                 context=STUDENT_CONTEXT, teacher_dim=T_D_MODEL):
        super().__init__()
        self.context = context
        self.zmax_enc = ZmaxEEGEncoder(in_ch=2, d_model=d_model, dropout=dropout)
        self.e4_enc   = E4Encoder(in_ch=3, d_model=d_model, dropout=dropout)
        self.zmax_agg = InterEpochAggregator(d_model=d_model, context=context)
        self.e4_agg   = InterEpochAggregator(d_model=d_model, context=context)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

        # Alignment head: fused embedding -> teacher embedding space
        self.align_proj = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Linear(d_model * 2, teacher_dim), nn.GELU(),
            nn.Linear(teacher_dim, teacher_dim),
        )

    def forward(self, zmax_x, e4_x):
        B, W, Cz, Tz = zmax_x.shape
        _, _, Ce, Te = e4_x.shape

        z_flat = self.zmax_enc(zmax_x.reshape(B * W, Cz, Tz)).view(B, W, -1)
        e_flat = self.e4_enc(e4_x.reshape(B * W, Ce, Te)).view(B, W, -1)

        z_ctx = self.zmax_agg(z_flat)
        e_ctx = self.e4_agg(e_flat)

        fused = torch.cat([z_ctx, e_ctx], dim=1)
        main_logits = self.classifier(fused)
        aligned_embedding = self.align_proj(fused)

        return main_logits, aligned_embedding


# ============================================================
# MMD LOSS (multi-kernel, unbiased estimator)
# ============================================================
def gaussian_kernel(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    """
    Computes a multi-scale Gaussian (RBF) kernel matrix over the
    concatenation of source and target samples.
    source: (n_s, d)   target: (n_t, d)
    Returns: (n_s+n_t, n_s+n_t) kernel matrix.
    """
    n_samples = source.size(0) + target.size(0)
    total = torch.cat([source, target], dim=0)

    total0 = total.unsqueeze(0).expand(int(total.size(0)), int(total.size(0)), int(total.size(1)))
    total1 = total.unsqueeze(1).expand(int(total.size(0)), int(total.size(0)), int(total.size(1)))
    l2_distance = ((total0 - total1) ** 2).sum(2)

    if fix_sigma:
        bandwidth = fix_sigma
    else:
        bandwidth = torch.sum(l2_distance.data) / (n_samples ** 2 - n_samples + 1e-8)

    bandwidth /= kernel_mul ** (kernel_num // 2)
    bandwidth_list = [bandwidth * (kernel_mul ** i) for i in range(kernel_num)]

    kernel_val = [torch.exp(-l2_distance / (bw + 1e-8)) for bw in bandwidth_list]
    return sum(kernel_val)


def mmd_loss(source, target, kernel_mul=MMD_KERNEL_MUL, kernel_num=MMD_KERNEL_NUM):
    """
    Standard biased MMD^2 estimator using multi-kernel MMD.
    source: student aligned embeddings (B, d)
    target: teacher center embeddings (B, d)
    Both must have the SAME batch size and feature dimension.
    """
    batch_size = source.size(0)
    kernels = gaussian_kernel(source, target, kernel_mul=kernel_mul, kernel_num=kernel_num)

    XX = kernels[:batch_size, :batch_size]
    YY = kernels[batch_size:, batch_size:]
    XY = kernels[:batch_size, batch_size:]
    YX = kernels[batch_size:, :batch_size]

    loss = torch.mean(XX) + torch.mean(YY) - torch.mean(XY) - torch.mean(YX)
    return loss


# ============================================================
# COMBINED DATASET (teacher window + student window + artifact_weight)
# ============================================================
class KDContextDataset(Dataset):
    def __init__(self, subject_list, psg_path, student_path,
                 teacher_context=TEACHER_CONTEXT, student_context=STUDENT_CONTEXT):
        self.teacher_context = teacher_context
        self.student_context = student_context
        self.data = []
        self.index = []

        skipped_mismatch = 0
        for sub in subject_list:
            psg_fp = os.path.join(psg_path, f"{sub}.npz")
            stu_fp = os.path.join(student_path, f"{sub}.npz")
            if not (os.path.exists(psg_fp) and os.path.exists(stu_fp)):
                continue

            with np.load(psg_fp) as d:
                eeg = d['eeg'][:, [0, 1], :]
                eog = d['eog'][:, [0], :]
                psg_signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                psg_labels = d['labels'].copy()

            with np.load(stu_fp) as d:
                zmax_arr = d['zmax_eeg']
                e4_arr   = d['e4']
                stu_labels = d['labels'].copy()
                art_w = d['artifact_weight']

            n = min(len(psg_labels), len(stu_labels), zmax_arr.shape[0], e4_arr.shape[0])
            if abs(len(psg_labels) - len(stu_labels)) > 5:
                skipped_mismatch += 1

            sub_idx = len(self.data)
            self.data.append((psg_signal, psg_labels, zmax_arr, e4_arr, art_w, stu_labels, n))
            for i in range(n):
                self.index.append((sub_idx, i, n))

        print(f"  Subjects loaded: {len(self.data)}  (large mismatches: {skipped_mismatch})")
        print(f"  Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n = self.index[idx]
        psg_signal, psg_labels, zmax_arr, e4_arr, art_w, stu_labels, _ = self.data[sub_idx]

        psg_window = []
        for offset in range(-self.teacher_context, self.teacher_context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            psg_window.append(psg_signal[ei])
        psg_x = np.stack(psg_window, axis=0)

        zmax_window, e4_window = [], []
        for offset in range(-self.student_context, self.student_context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            zmax_window.append(zmax_arr[ei])
            e4_window.append(e4_arr[ei])
        zmax_x = np.stack(zmax_window, axis=0)
        e4_x   = np.stack(e4_window, axis=0)

        aw = float(art_w[center_i]) if center_i < len(art_w) else 1.0
        y  = int(stu_labels[center_i]) if center_i < len(stu_labels) else int(psg_labels[center_i])

        return (
            torch.FloatTensor(psg_x),
            torch.FloatTensor(zmax_x),
            torch.FloatTensor(e4_x),
            torch.tensor(aw, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
        )


# ============================================================
# 3-WAY SPLIT + BUILD DATASETS
# ============================================================
_train_path = os.path.join(SPLIT_PATH, "_train_subs.npy")
_val_path   = os.path.join(SPLIT_PATH, "_val_subs.npy")
_test_path  = os.path.join(SPLIT_PATH, "_test_subs.npy")

for p, name in [(_train_path, "_train_subs.npy"),
                (_val_path,   "_val_subs.npy"),
                (_test_path,  "_test_subs.npy")]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{name} not found at {SPLIT_PATH}")

TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
VAL_SUBS   = np.load(_val_path,   allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()

assert set(TRAIN_SUBS).isdisjoint(VAL_SUBS),  "TRAIN/VAL subject overlap!"
assert set(TRAIN_SUBS).isdisjoint(TEST_SUBS), "TRAIN/TEST subject overlap!"
assert set(VAL_SUBS).isdisjoint(TEST_SUBS),   "VAL/TEST subject overlap!"

print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Val:{len(VAL_SUBS)}  Test:{len(TEST_SUBS)}")

print("\nBuilding context-window KD datasets...")
print("Train:")
train_ds = KDContextDataset(TRAIN_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print("Val:")
val_ds   = KDContextDataset(VAL_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)
print("Test:")
test_ds  = KDContextDataset(TEST_SUBS, PSG_EEG_PATH, STUDENT_DATA_PATH)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    if len(ds) == 0:
        raise RuntimeError(f"{name}_ds has 0 samples! Check PSG_EEG_PATH / STUDENT_DATA_PATH.")
print("Datasets ready.")


# ============================================================
# LOAD FROZEN TEACHER ENSEMBLE
# ============================================================
print(f"\nLoading {len(TEACHER_SEEDS)} frozen teacher checkpoints from {TEACHER_CKPT_DIR}...")
teachers = []
for seed in TEACHER_SEEDS:
    ckpt = os.path.join(TEACHER_CKPT_DIR, f"best_seed{seed}.pt")
    if not os.path.exists(ckpt):
        print(f"  MISSING: {ckpt}")
        continue
    m = MultiScaleSleepNetPlain(in_ch=3, context=TEACHER_CONTEXT, window=TEACHER_WINDOW).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    teachers.append(m)
print(f"Loaded {len(teachers)} teacher models (frozen).")
if len(teachers) == 0:
    raise RuntimeError(f"No teacher checkpoints found in {TEACHER_CKPT_DIR}.")


@torch.no_grad()
def teacher_soft_targets_and_feature(psg_x, temperature=KD_TEMPERATURE):
    """
    Returns:
      probs_avg   : (B, 5) averaged softmax over all 5 teacher seeds (for KD)
      feat_avg    : (B, T_D_MODEL) averaged center embedding over all 5
                    teacher seeds (for MMD alignment)
    """
    probs_sum = None
    feat_sum = None
    for m in teachers:
        logits, feat = m.forward_with_feature(psg_x)
        probs = F.softmax(logits / temperature, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
        feat_sum = feat if feat_sum is None else feat_sum + feat
    return probs_sum / len(teachers), feat_sum / len(teachers)


# ============================================================
# CLASS WEIGHTS -- from TRAIN only
# ============================================================
all_train_labels = []
for _, _, _, _, _, labs, _ in train_ds.data:
    all_train_labels.extend(labs.tolist())
label_counts = np.array([Counter(all_train_labels).get(i, 1) for i in range(5)], dtype=np.float32)
cw = torch.FloatTensor(label_counts.sum() / (5 * label_counts)).to(device)
print(f"\nClass weights (from TRAIN set only): {dict(zip(LABEL_NAMES, cw.cpu().numpy().round(3)))}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# LOSS: main artifact-weighted CE + KD (identical to S5) PLUS
# an MMD alignment term between student and teacher embeddings.
# ============================================================
def compute_loss(main_logits, aligned_embedding, teacher_probs, teacher_feat,
                  labels, artifact_weight, temperature, lam_kd, lam_mmd, class_weights):

    student_log_probs = F.log_softmax(main_logits / temperature, dim=1)
    kd = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * (temperature ** 2)

    ce_per_sample = F.cross_entropy(main_logits, labels, weight=class_weights, reduction='none')
    weighted_ce = ce_per_sample * artifact_weight
    valid = artifact_weight > 0
    ce = weighted_ce[valid].mean() if valid.sum() > 0 else weighted_ce.mean()

    mmd = mmd_loss(aligned_embedding, teacher_feat)

    total = ce + lam_kd * kd + lam_mmd * mmd
    return total, kd.item(), ce.item(), mmd.item()


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = total_kd = total_ce = total_mmd = 0
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, aw, y in loader:
        psg_x, zmax_x, e4_x, aw, y = (
            psg_x.to(device), zmax_x.to(device), e4_x.to(device),
            aw.to(device), y.to(device)
        )
        optimizer.zero_grad()
        teacher_probs, teacher_feat = teacher_soft_targets_and_feature(psg_x)
        main_logits, aligned_embedding = model(zmax_x, e4_x)

        loss, kd_val, ce_val, mmd_val = compute_loss(
            main_logits, aligned_embedding, teacher_probs, teacher_feat,
            y, aw, KD_TEMPERATURE, KD_LAMBDA, MMD_LAMBDA, cw
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item(); total_kd += kd_val; total_ce += ce_val; total_mmd += mmd_val
        preds.extend(main_logits.argmax(1).cpu().numpy())
        labs_all.extend(y.cpu().numpy())

    n = len(loader)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    return total_loss / n, total_ce / n, total_kd / n, total_mmd / n, acc, f1


@torch.no_grad()
def evaluate(model, loader):
    """Uses ONLY the main classifier -- alignment head is never
    used at inference, so this is directly comparable to S5."""
    model.eval()
    preds, labs_all = [], []
    for psg_x, zmax_x, e4_x, aw, y in loader:
        zmax_x, e4_x = zmax_x.to(device), e4_x.to(device)
        main_logits, _ = model(zmax_x, e4_x)
        preds.extend(main_logits.argmax(1).cpu().numpy())
        labs_all.extend(y.numpy())
    preds, labs_all = np.array(preds), np.array(labs_all)
    acc = accuracy_score(labs_all, preds)
    f1 = f1_score(labs_all, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs_all, preds)
    per_cls = f1_score(labs_all, preds, average=None, zero_division=0, labels=[0, 1, 2, 3, 4])
    return acc, f1, kappa, per_cls


# ============================================================
# CSV SETUP
# ============================================================
epoch_csv_path = os.path.join(EVAL_PATH, "epoch_log.csv")
epoch_fields = [
    "seed", "epoch", "train_loss", "train_ce", "train_kd", "train_mmd",
    "train_acc", "train_f1_macro",
    "val_acc", "val_f1_macro", "val_kappa",
    "val_f1_Wake", "val_f1_N1", "val_f1_N2", "val_f1_N3", "val_f1_REM",
    "lr", "is_best",
]
with open(epoch_csv_path, 'w', newline='') as f:
    csv.DictWriter(f, epoch_fields).writeheader()

csv_path = os.path.join(EVAL_PATH, "summary.csv")
fields = ["seed", "best_epoch", "best_val_f1", "acc", "f1_macro", "kappa",
          "f1_Wake", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fields).writeheader()

all_results = []
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSTUDENT SEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=0, generator=torch.Generator().manual_seed(seed))
    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = DomainAlignedStudentModel(context=STUDENT_CONTEXT).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Student parameters: {n_params:,}  (teacher frozen, not counted)")

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4, betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_val_f1 = 0.0
    best_epoch  = -1
    best_path = os.path.join(EVAL_PATH, f"best_student_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_ce, tr_kd, tr_mmd, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler)
        vl_acc, vl_f1, vl_kap, vl_per = evaluate(model, val_loader)

        is_best = 0
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_epoch  = epoch
            torch.save(model.state_dict(), best_path)
            is_best = 1

        tag = " <- BEST" if is_best else ""
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f}(CE:{tr_ce:.3f}+KD:{tr_kd:.3f}+MMD:{tr_mmd:.4f}) "
              f"TrF1:{tr_f1:.3f} | ValAcc:{vl_acc:.3f} ValF1:{vl_f1:.3f} Valk:{vl_kap:.3f}{tag}")

        with open(epoch_csv_path, 'a', newline='') as f:
            csv.DictWriter(f, epoch_fields).writerow({
                "seed": seed, "epoch": epoch,
                "train_loss": round(tr_loss, 6), "train_ce": round(tr_ce, 6),
                "train_kd": round(tr_kd, 6), "train_mmd": round(tr_mmd, 6),
                "train_acc": round(tr_acc, 6), "train_f1_macro": round(tr_f1, 6),
                "val_acc": round(vl_acc, 6), "val_f1_macro": round(vl_f1, 6), "val_kappa": round(vl_kap, 6),
                "val_f1_Wake": round(vl_per[0], 6), "val_f1_N1": round(vl_per[1], 6),
                "val_f1_N2": round(vl_per[2], 6), "val_f1_N3": round(vl_per[3], 6),
                "val_f1_REM": round(vl_per[4], 6),
                "lr": optimizer.param_groups[0]['lr'], "is_best": is_best,
            })

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate(model, test_loader)

    print(f"\n  Seed {seed}: best val F1={best_val_f1:.4f} at epoch {best_epoch}")
    print(f"  Seed {seed} FINAL TEST (evaluated once): Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap, 'per_cls': fin_per,
                         'best_epoch': best_epoch, 'best_val_f1': best_val_f1})
    with open(csv_path, 'a', newline='') as f:
        csv.DictWriter(f, fields).writerow({
            "seed": seed, "best_epoch": best_epoch, "best_val_f1": round(best_val_f1, 4),
            "acc": round(fin_acc, 4), "f1_macro": round(fin_f1, 4), "kappa": round(fin_kap, 4),
            "f1_Wake": round(fin_per[0], 4), "f1_N1": round(fin_per[1], 4),
            "f1_N2": round(fin_per[2], 4), "f1_N3": round(fin_per[3], 4),
            "f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs = np.array([r['acc'] for r in all_results]) * 100
f1s = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])
n1s = np.array([r['per_cls'][1] for r in all_results])

print(f"\n{'='*60}\nS6c: DOMAIN-AWARE ALIGNMENT (MMD) -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")
print(f"N1 F1    : {n1s.mean():.4f} +- {n1s.std():.4f}")

print(f"\n{'='*60}\nCOMPARE AGAINST S5 BASELINE (no domain alignment)\n{'='*60}")
print(f"{'Model':<45} {'Acc%':>8} {'F1':>8} {'N1 F1':>8}")
print("-" * 70)
print(f"{'S5 (context-window, no MMD alignment)':<45} {'65.98':>8} {'0.5974':>8} {'0.3514':>8}")
print(f"{'S6a (context-window, constant KD lambda)':<45} {'67.52':>8} {'0.6127':>8} {'0.3651':>8}")
print(f"{'S6b (context-window, (adaptive KD)':<45} {'65.88':>8} {'0.5934':>8} {' 0.3495':>8}")
print(f"{'S6c (domain-aware MMD alignment, this run)':<45} {accs.mean():>8.2f} {f1s.mean():>8.4f} {n1s.mean():>8.4f}")
print("=" * 70)

diff_f1 = f1s.mean() - 0.5974
if diff_f1 > 0:
    print(f"\nS6c beats S5 by {diff_f1:+.4f} F1 -- domain-aware MMD alignment helped.")
    print("Worth keeping / reporting as an improvement.")
else:
    print(f"\nS6c does NOT beat S5 ({diff_f1:+.4f} F1). Report S5 as the final")
    print("result, and this as a negative/neutral ablation: explicit")
    print("representation-space alignment did not outweigh the added")
    print("training complexity/noise for this dataset size.")
print("\nNote: if MMD_LAMBDA=0.1 shows no effect either way, try sweeping")
print("MMD_LAMBDA in {0.01, 0.05, 0.5} using VAL F1 (not TEST) to pick the")
print("best value before drawing conclusions -- MMD is somewhat sensitive")
print("to this weight.")

print(f"\nEpoch log : {epoch_csv_path}")
print(f"Summary   : {csv_path}")
print("Done!")

Device        : cuda
S6c: DOMAIN-AWARE REPRESENTATION ALIGNMENT (MMD)
Student context: +-5 (window=11)
KD lambda     : 0.5   Temperature: 4.0   MMD lambda: 0.1
Output        : D:\22\AA\AA journal\evaluation\teacher-student\s6c_domain_aware_mmd_studentC5
Split loaded -> Train:65  Val:11  Test:20

Building context-window KD datasets...
Train:
  Subjects loaded: 61  (large mismatches: 0)
  Samples: 57,021
Val:
  Subjects loaded: 10  (large mismatches: 0)
  Samples: 9,742
Test:
  Subjects loaded: 19  (large mismatches: 0)
  Samples: 18,898
Datasets ready.

Loading 5 frozen teacher checkpoints from D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed...
Loaded 5 teacher models (frozen).

Class weights (from TRAIN set only): {'Wake': np.float32(2.094), 'N1': np.float32(3.192), 'N2': np.float32(0.436), 'N3': np.float32(1.0), 'REM': np.float32(1.092)}

STUDENT SEED 42  (1/5)
  Student parameters: 280,069  (teacher frozen, not counted)
  Ep[01/30] Loss:4.498(CE:1.058+KD:6.784+MMD:0.4747)